# 🏗️ Projet 10 — Fine-tuning BERT NER pour le BTP (Google Colab)

Ce notebook entraîne **DistilBERT** à extraire automatiquement les entités des descriptions de travaux :
`TYPE_TRAVAIL`, `SURFACE`, `MATERIAU`, `ACTIVITE_SPECIALE`, `BUDGET`, `NB_OUVRIERS`, `DUREE_ESTIMEE`.

**Avant de lancer :** menu `Exécution` → `Modifier le type d'exécution` → choisis **GPU (T4)**.

Puis `Exécution` → `Tout exécuter`. À la fin, un fichier `.zip` du modèle se télécharge automatiquement.

## 1. Vérifier le GPU

In [ ]:
import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Carte :', torch.cuda.get_device_name(0))
else:
    print('⚠️ Pas de GPU ! Va dans Exécution > Modifier le type d\'exécution > GPU')

## 2. Installer les dépendances

In [ ]:
!pip install -q transformers==4.35.2 scikit-learn seaborn

## 3. Charger le dataset depuis GitHub
Le dataset (500 exemples) est récupéré directement depuis ta branche `bert-finetuning`.

In [ ]:
import urllib.request, json

URL = 'https://raw.githubusercontent.com/adamcode-tech/Artisandispatch/bert-finetuning/entrainement%20machine%20learning/data/ner_dataset.json'
urllib.request.urlretrieve(URL, 'ner_dataset.json')

with open('ner_dataset.json', 'r', encoding='utf-8') as f:
    dataset_raw = json.load(f)

dataset = dataset_raw['dataset']
ENTITY_TYPES = dataset_raw['metadata']['entity_types']
print(f'✓ {len(dataset)} exemples chargés')
print(f'✓ Entités : {ENTITY_TYPES}')
print('\nExemple :', dataset[0])

## 4. Préparer les tokens et labels (format NER)

In [ ]:
from transformers import DistilBertTokenizer
from tqdm.auto import tqdm

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128

# Mappings label <-> id
label2id = {'O': 0}
id2label = {0: 'O'}
lid = 1
for ent in ENTITY_TYPES:
    label2id[f'B-{ent}'] = lid; id2label[lid] = f'B-{ent}'; lid += 1
print(f'{len(label2id)} labels :', label2id)

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

prepared = []
for ex in tqdm(dataset):
    text, entities = ex['text'], ex['entities']
    enc = tokenizer(text, max_length=MAX_LENGTH, padding='max_length', truncation=True, return_tensors='pt')
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    labels = [label2id['O']] * len(tokens)
    # Marquer les entités
    for etype, evalue in entities.items():
        etoks = tokenizer.tokenize(evalue.lower())
        for i in range(len(tokens) - len(etoks) + 1):
            if all(tokens[i+j].lower().replace('##','') == etoks[j].lower().replace('##','') for j in range(len(etoks))):
                labels[i] = label2id.get(f'B-{etype}', label2id['O'])
                break
    # Ignorer [CLS], [SEP], [PAD]
    labels[0] = -100
    for i in range(len(tokens)-1, -1, -1):
        if tokens[i] == '[SEP]':
            labels[i] = -100; break
    for i in range(len(tokens)):
        if tokens[i] == '[PAD]':
            labels[i] = -100
    prepared.append({
        'input_ids': enc['input_ids'].squeeze(),
        'attention_mask': enc['attention_mask'].squeeze(),
        'labels': torch.tensor(labels)
    })
print('✓ Données préparées :', len(prepared))

## 5. Créer les DataLoaders (train / val / test)

In [ ]:
from torch.utils.data import DataLoader, Dataset

BATCH_SIZE = 16
n_train = int(0.7*len(prepared)); n_val = int(0.15*len(prepared))
train_data = prepared[:n_train]
val_data   = prepared[n_train:n_train+n_val]
test_data  = prepared[n_train+n_val:]

class NERDataset(Dataset):
    def __init__(self, d): self.d = d
    def __len__(self): return len(self.d)
    def __getitem__(self, i): return self.d[i]

train_loader = DataLoader(NERDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(NERDataset(val_data),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(NERDataset(test_data),  batch_size=BATCH_SIZE)
print(f'Train {len(train_data)} | Val {len(val_data)} | Test {len(test_data)}')

## 6. Charger DistilBERT et entraîner (5 epochs)

In [ ]:
from transformers import DistilBertForTokenClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 5

model = DistilBertForTokenClassification.from_pretrained(MODEL_NAME, num_labels=len(label2id)).to(device)
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader)*EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

def evaluate(loader):
    model.eval(); preds, labs = [], []
    with torch.no_grad():
        for b in loader:
            out = model(input_ids=b['input_ids'].to(device), attention_mask=b['attention_mask'].to(device))
            p = torch.argmax(out.logits, dim=2).cpu().numpy()
            preds.append(p); labs.append(b['labels'].numpy())
    preds = np.concatenate(preds).flatten(); labs = np.concatenate(labs).flatten()
    mask = labs != -100
    return accuracy_score(labs[mask], preds[mask]), f1_score(labs[mask], preds[mask], average='weighted', zero_division=0)

train_losses, val_accs, val_f1s = [], [], []
for epoch in range(EPOCHS):
    model.train(); tot = 0
    for b in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        optimizer.zero_grad()
        out = model(input_ids=b['input_ids'].to(device), attention_mask=b['attention_mask'].to(device), labels=b['labels'].to(device))
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); tot += out.loss.item()
    tl = tot/len(train_loader); acc, f1 = evaluate(val_loader)
    train_losses.append(tl); val_accs.append(acc); val_f1s.append(f1)
    print(f'  Loss {tl:.4f} | Val Acc {acc:.4f} | Val F1 {f1:.4f}')

test_acc, test_f1 = evaluate(test_loader)
print(f'\n📊 TEST — Accuracy {test_acc*100:.2f}% | F1 {test_f1:.4f}')

## 7. Tester sur des exemples réels

In [ ]:
examples = [
    'Remplacement toiture 150m² ardoise, désamiantage, budget 25000€',
    'Électricité 4 pièces, mise aux normes, équipe de 3 personnes',
    'Maçonnerie extension 80m² béton, 35 jours',
    'Peinture intérieure 200m², budget 8000€',
]
model.eval()
for ex in examples:
    enc = tokenizer(ex, max_length=MAX_LENGTH, padding='max_length', truncation=True, return_tensors='pt')
    with torch.no_grad():
        out = model(input_ids=enc['input_ids'].to(device), attention_mask=enc['attention_mask'].to(device))
    preds = torch.argmax(out.logits, dim=2)[0].cpu()
    toks = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    found = {}
    for t, p in zip(toks, preds):
        lab = id2label.get(int(p), 'O')
        if t in ['[CLS]','[SEP]','[PAD]'] or lab == 'O': continue
        et = lab.split('-')[1]
        found.setdefault(et, []).append(t.replace('##',''))
    print(f'\n📝 {ex}')
    for et, vals in found.items():
        print(f'   • {et} : {" ".join(vals)}')

## 8. Graphiques d'entraînement

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(range(1, EPOCHS+1), train_losses, 'b-o'); ax[0].set_title('Training Loss'); ax[0].grid(alpha=.3)
ax[1].plot(range(1, EPOCHS+1), val_accs, 'g-o'); ax[1].set_title('Val Accuracy'); ax[1].set_ylim(0,1); ax[1].grid(alpha=.3)
ax[2].plot(range(1, EPOCHS+1), val_f1s, 'orange', marker='o'); ax[2].set_title('Val F1'); ax[2].set_ylim(0,1); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.savefig('resultats.png', dpi=150); plt.show()

## 9. Sauvegarder et télécharger le modèle 📥

In [ ]:
import json
model.save_pretrained('distilbert_btp_ner')
tokenizer.save_pretrained('distilbert_btp_ner')
with open('distilbert_btp_ner/label_mappings.json', 'w') as f:
    json.dump({'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}}, f)

!zip -r distilbert_btp_ner.zip distilbert_btp_ner

from google.colab import files
files.download('distilbert_btp_ner.zip')
print('✅ Modèle téléchargé ! Décompresse-le et utilise-le dans ton app.')